# Plotly Dashboard — Enhanced

This notebook contains the cleaned & enhanced Plotly Dash dashboard for your cricket analytics project. Run the final code cell to start the dashboard locally (it will print the URL and open at `http://127.0.0.1:8050`).

Make sure project files are accessible (set `AI_PROJECT_ROOT` environment variable if needed).

In [1]:
"""
Enhanced Plotly Dash Dashboard (fixed)
Place this file at project root (same parent as `data/` and `models/`).
Requires: dash, dash-bootstrap-components, pandas, plotly, joblib, scikit-learn
This version includes fixes for:
 - phase_sr name normalization and robust matching
 - stable get_recommendation implementation
 - recommend_optimal_strategy that builds feature vectors using model.feature_names_in_
 - safer defaults (column means) for missing features so model predictions vary
 - removed stray/undefined variables and improved error handling
"""

import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import joblib
import warnings
import traceback

warnings.filterwarnings("ignore")

# ---------- Configuration ----------
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    cwd = Path(os.getcwd()).resolve()
    if cwd.name.lower() == "notebooks":
        PROJECT_ROOT = cwd.parent
    else:
        PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
FEATURE_DATA_DIR = PROJECT_ROOT / "model_feature_engineered_datasets"

FILES = {
    "final": DATA_DIR / "final_processed_data.csv",
    "batsman_profiles": DATA_DIR / "batsman_profiles.csv",
    "batsman_similarity": DATA_DIR / "batsman_similarity.csv",
    "bowler_stats": DATA_DIR / "bowler_stats.csv",
    "bowling_success": DATA_DIR / "bowling_success_model.csv",
    "phase_sr": DATA_DIR / "phase_sr.csv",
    "batsman_stats": DATA_DIR / "batsman_stats.csv",
}

# ---------- Utilities ----------

def safe_csv_load(path: Path):
    if path is None or not path.exists():
        print(f"[warn] file not found: {path}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        print(f"[info] Loaded {path} -> {df.shape}")
        return df
    except Exception as e:
        print(f"[warn] Failed loading {path}: {e}")
        return pd.DataFrame()


def safe_joblib_load(path: Path):
    if path is None or not path.exists():
        print(f"[warn] model not found: {path}")
        return None
    try:
        model = joblib.load(path)
        print(f"[info] Loaded model {path}")
        return model
    except Exception as e:
        print(f"[warn] Failed loading model {path}: {e}")
        return None


# ---------- Load feature-engineered sets (if present) ----------
MODEL_DATASETS = {
    "good_ball": safe_csv_load(FEATURE_DATA_DIR / "df_model_good_ball.csv"),
    "ball_type": safe_csv_load(FEATURE_DATA_DIR / "df_model_ball_type.csv"),
    "boundary": safe_csv_load(FEATURE_DATA_DIR / "df_model_boundary.csv"),
    "line": safe_csv_load(FEATURE_DATA_DIR / "df_model_line.csv"),
}

# ---------- Load raw datasets ----------
df_final = safe_csv_load(FILES["final"])
batsman_profiles = safe_csv_load(FILES["batsman_profiles"])

# similarity may be square CSV with index col 0 — try robust load
batsman_similarity = pd.DataFrame()
if FILES["batsman_similarity"].exists():
    try:
        batsman_similarity = pd.read_csv(FILES["batsman_similarity"], index_col=0)
        print(f"[info] Loaded {FILES['batsman_similarity']} -> {batsman_similarity.shape}")
    except Exception as e:
        print(f"[warn] reading batsman_similarity with index_col failed: {e}")
        batsman_similarity = safe_csv_load(FILES["batsman_similarity"])
else:
    batsman_similarity = safe_csv_load(FILES["batsman_similarity"])

if not batsman_similarity.empty:
    try:
        batsman_similarity.index = batsman_similarity.index.astype(str).str.strip().str.title()
        batsman_similarity.columns = batsman_similarity.columns.astype(str).str.strip().str.title()
    except Exception:
        pass

bowler_stats = safe_csv_load(FILES["bowler_stats"])
bowling_success = safe_csv_load(FILES["bowling_success"])
phase_sr = safe_csv_load(FILES["phase_sr"])

# ---------- Normalize and ensure Batsman_Name exists ----------

def normalize_columns(df):
    if df is None or df.empty:
        return df
    df = df.copy()
    colmap = {}
    if "Batsman_Name_clean" in df.columns and "Batsman_Name" not in df.columns:
        colmap["Batsman_Name_clean"] = "Batsman_Name"
    if "run" in df.columns and "Runs" not in df.columns:
        colmap["run"] = "Runs"
    if "Bowler_Name" in df.columns and "Bowler" not in df.columns:
        colmap["Bowler_Name"] = "Bowler"
    df.rename(columns=colmap, inplace=True)
    return df

for name in ["df_final", "batsman_profiles", "bowler_stats", "phase_sr"]:
    df_obj = globals().get(name)
    globals()[name] = normalize_columns(df_obj)

# ensure Batsman_Name present
if isinstance(df_final, pd.DataFrame) and 'Batsman_Name' in df_final.columns:
    df_final['Batsman_Name'] = df_final['Batsman_Name'].astype(str).str.strip().str.title()
else:
    for alt in ['Full Name', 'Full_Name', 'FullName', 'Batsman', 'Batsman_Name_clean']:
        if isinstance(df_final, pd.DataFrame) and alt in df_final.columns:
            df_final['Batsman_Name'] = df_final[alt].astype(str).str.strip().str.title()
            break

# ---------- Load models ----------
predict_good_ball_model = safe_joblib_load(MODELS_DIR / "predict_good_ball_model_random_forest.joblib")
pitch_line_model = safe_joblib_load(MODELS_DIR / "pitch_line_model.joblib")
pitch_length_model = safe_joblib_load(MODELS_DIR / "pitch_length_model.joblib")
model_balltype = safe_joblib_load(MODELS_DIR / "model_balltype.joblib")
predict_boundary_model = safe_joblib_load(MODELS_DIR / "predict_boundary_model.joblib")

# Try again to ensure variables set (harmless if None)
pitch_line_model = pitch_line_model or safe_joblib_load(MODELS_DIR / "pitch_line_model.joblib")
pitch_length_model = pitch_length_model or safe_joblib_load(MODELS_DIR / "pitch_length_model.joblib")

# label encoder for bowler styles
le_bowler = None
if (MODELS_DIR / "le_bowler.joblib").exists():
    try:
        le_bowler = joblib.load(MODELS_DIR / "le_bowler.joblib")
        print("[info] Loaded bowler label encoder")
    except Exception as e:
        print("[warn] Failed loading le_bowler.joblib:", e)

from sklearn.preprocessing import LabelEncoder
bowling_types = []
if "line" in MODEL_DATASETS and not MODEL_DATASETS["line"].empty and "Bowler_Bowling_Style" in MODEL_DATASETS["line"].columns:
    bowling_types = sorted(MODEL_DATASETS["line"]["Bowler_Bowling_Style"].dropna().unique().tolist())
elif isinstance(df_final, pd.DataFrame) and "Bowler_Bowling_Style" in df_final.columns:
    bowling_types = sorted(df_final["Bowler_Bowling_Style"].dropna().unique().tolist())

if le_bowler is None:
    try:
        le_bowler = LabelEncoder()
        if bowling_types:
            le_bowler.fit(bowling_types)
        else:
            fallback = ["right-arm fast", "right-arm medium-fast", "left-arm medium-fast", "legbreak", "offbreak", "slow left-arm orthodox"]
            le_bowler.fit(fallback)
        print("[info] Created in-memory le_bowler")
    except Exception as e:
        print("[warn] Could not create le_bowler:", e)
        le_bowler = None

# Build pitch mappings from MODEL_DATASETS['line'] if available
pitchLine_mapping = {}
pitchLength_mapping = {}
df_line = MODEL_DATASETS.get("line", pd.DataFrame())
if not df_line.empty:
    if 'pitchLine' in df_line.columns and 'pitchLine_code' in df_line.columns:
        unique_pairs = df_line[[ 'pitchLine_code', 'pitchLine']].drop_duplicates()
        pitchLine_mapping = dict(zip(unique_pairs['pitchLine_code'].astype(int), unique_pairs['pitchLine'].astype(str)))
    if 'pitchLength' in df_line.columns and 'pitchLength_code' in df_line.columns:
        unique_pairs = df_line[[ 'pitchLength_code', 'pitchLength']].drop_duplicates()
        pitchLength_mapping = dict(zip(unique_pairs['pitchLength_code'].astype(int), unique_pairs['pitchLength'].astype(str)))

if not pitchLine_mapping:
    pitchLine_mapping = {0: "ON_THE_STUMPS", 1: "OUTSIDE_OFF", 2: "OUTSIDE_LEG", 3: "LEG_STUMP", 4: "WIDE_OUTSIDE_OFFSTUMP", 5: "DOWN_LEG"}
if not pitchLength_mapping:
    pitchLength_mapping = {0: "FULL", 1: "GOOD_LENGTH", 2: "SHORT_OF_A_GOOD_LENGTH", 3: "YORKER", 4: "SHORT", 5: "BOUNCER"}

# Derive feature_cols from df_line or fallback
feature_cols = []
if not df_line.empty:
    numeric_cols = df_line.select_dtypes(include='number').columns.tolist()
    for lab in ["pitchLine_code", "pitchLength_code"]:
        if lab in numeric_cols:
            numeric_cols.remove(lab)
    feature_cols = numeric_cols
else:
    feature_cols = ["strike_rate", "boundary_rate", "dot_ball_pct", "consistency_index", "pressure_index_batsman", "oversActual"]

print("[info] feature_cols (len={}) : {}".format(len(feature_cols), feature_cols))

# ---------- Helpers ----------

def normalize_name_simple(s):
    if s is None:
        return ""
    s = str(s).lower()
    s = s.replace('.', ' ').replace('_', ' ').strip()
    s = ''.join(ch for ch in s if ch.isalpha() or ch.isspace())
    return ' '.join(s.split())


def normalize_df_name_column(df, possible_name_cols=None):
    """Return df and name_col used (title-cased)"""
    if df is None or df.empty:
        return df, None
    if possible_name_cols is None:
        possible_name_cols = [c for c in df.columns if 'name' in c.lower()]
    if not possible_name_cols:
        return df, None
    name_col = possible_name_cols[0]
    df = df.copy()
    df[name_col] = df[name_col].astype(str).apply(lambda x: normalize_name_simple(x)).str.title()
    return df, name_col

# Normalize phase_sr name column if present
if isinstance(phase_sr, pd.DataFrame) and not phase_sr.empty:
    phase_sr, _ = normalize_df_name_column(phase_sr)

# normalize batsman_group later after load

# ---------- recommend_optimal_strategy (improved) ----------

def get_model_feature_names(model, fallback_df=None):
    if model is None:
        return []
    try:
        if hasattr(model, 'feature_names_in_'):
            return list(model.feature_names_in_)
    except Exception:
        pass
    if fallback_df is not None and not fallback_df.empty:
        return fallback_df.select_dtypes(include='number').columns.tolist()
    return []


def recommend_optimal_strategy(batsman_name, phase):
    """
    Build a one-row DataFrame matching the models' expected feature names.
    Fill missing numeric features with column means from MODEL_DATASETS['line'] or df_final.
    """
    if batsman_name is None or str(batsman_name).strip() == "":
        return {"Error": "No batsman specified."}

    # build canonical batsman_row
    batsman_row = None
    if 'batsman_group' in globals() and isinstance(batsman_group, pd.DataFrame) and not batsman_group.empty:
        batsman_row = match_batsman(batsman_name, batsman_group)
    if batsman_row is None and isinstance(batsman_profiles, pd.DataFrame) and not batsman_profiles.empty:
        batsman_row = match_batsman(batsman_name, batsman_profiles)
    if batsman_row is None and isinstance(df_final, pd.DataFrame) and not df_final.empty:
        mask = df_final['Batsman_Name'].astype(str).str.contains(str(batsman_name), case=False, na=False)
        if mask.any():
            batsman_row = df_final.loc[mask].iloc[0].to_dict()

    # Prepare feature names
    line_feats = get_model_feature_names(pitch_line_model, df_line if not df_line.empty else df_final)
    length_feats = get_model_feature_names(pitch_length_model, df_line if not df_line.empty else df_final)
    required = list(dict.fromkeys((line_feats or []) + (length_feats or [])))

    # If no required features inferred, fallback to feature_cols
    if not required:
        required = list(feature_cols)

    # Build defaults from MODEL_DATASETS['line'] or df_final
    stats_source = df_line if (not df_line.empty) else (df_final if isinstance(df_final, pd.DataFrame) and not df_final.empty else None)
    defaults = {}
    if stats_source is not None and not stats_source.empty:
        for c in required:
            if c in stats_source.columns and pd.api.types.is_numeric_dtype(stats_source[c]):
                defaults[c] = float(stats_source[c].mean())
            else:
                defaults[c] = 0.0
    else:
        for c in required:
            defaults[c] = 0.0

    # try to populate from batsman_row
    sample = defaults.copy()
    try:
        if batsman_row is not None:
            for c in required:
                # tolerant key lookup
                if isinstance(batsman_row, (pd.Series, dict)):
                    if c in batsman_row:
                        try:
                            sample[c] = float(batsman_row[c])
                            continue
                        except Exception:
                            pass
                    # try lower-case key
                    lc = c.lower()
                    if lc in batsman_row:
                        try:
                            sample[c] = float(batsman_row[lc])
                            continue
                        except Exception:
                            pass
                # phase-specific SR override
                if 'strike' in c.lower() or 'sr' in c.lower():
                    # use phase_sr if available
                    try:
                        if isinstance(phase_sr, pd.DataFrame) and not phase_sr.empty:
                            pname_col = next((col for col in phase_sr.columns if 'name' in col.lower()), None)
                            psr_col = next((col for col in phase_sr.columns if 'strike' in col.lower() or 'sr' in col.lower()), None)
                            pphase_col = next((col for col in phase_sr.columns if 'phase' in col.lower()), None)
                            if pname_col and psr_col:
                                sub = phase_sr[phase_sr[pname_col].astype(str).str.contains(str(batsman_name), case=False, na=False)]
                                if not sub.empty:
                                    if pphase_col is not None:
                                        row = sub[sub[pphase_col].astype(str).str.strip().str.title() == str(phase).strip().title()]
                                        if row.empty:
                                            row = sub.iloc[[0]]
                                    else:
                                        row = sub.iloc[[0]]
                                    val = row[psr_col].iloc[0]
                                    sample[c] = float(val) if pd.notna(val) else sample[c]
                    except Exception:
                        pass
    except Exception:
        pass

    # Build DataFrame and predict for each bowling style available
    bowling_styles = []
    if isinstance(df_final, pd.DataFrame) and 'Bowler_Bowling_Style' in df_final.columns:
        bowling_styles = list(df_final['Bowler_Bowling_Style'].dropna().astype(str).unique())
    if not bowling_styles and isinstance(bowler_stats, pd.DataFrame) and not bowler_stats.empty:
        col = _find_column(bowler_stats, ['Bowler_Bowling_Style', 'Bowler Style', 'Bowling_Style'])
        if col:
            bowling_styles = list(bowler_stats[col].dropna().astype(str).unique())
    if not bowling_styles:
        bowling_styles = ["right-arm fast", "right-arm medium-fast", "left-arm medium-fast", "legbreak"]

    results = []
    for style in bowling_styles:
        row = sample.copy()
        # encode bowler style if model expects it
        try:
            if le_bowler is not None:
                enc_col_candidates = [c for c in required if 'bowler' in c.lower() or 'bowling' in c.lower()]
                if enc_col_candidates:
                    code = 0
                    try:
                        code = int(le_bowler.transform([style])[0])
                    except Exception:
                        # fallback mapping via categorical codes in df_final
                        if isinstance(df_final, pd.DataFrame) and 'Bowler_Bowling_Style' in df_final.columns:
                            cats = pd.Categorical(df_final['Bowler_Bowling_Style'].astype(str)).categories.tolist()
                            code = cats.index(style) if style in cats else len(cats)
                    for ec in enc_col_candidates:
                        row[ec] = code
                else:
                    # try common name
                    row['Bowler_Bowling_Style_encoded'] = int(le_bowler.transform([style])[0]) if le_bowler is not None else 0
        except Exception:
            pass

        # Build X aligned to model feature names (use pitch_line_model's feature names if available)
        try:
            if hasattr(pitch_line_model, 'feature_names_in_'):
                X_line = pd.DataFrame([{k: row.get(k, 0.0) for k in pitch_line_model.feature_names_in_}])
            else:
                X_line = pd.DataFrame([row])[required].fillna(0)
        except Exception:
            X_line = pd.DataFrame([row])[required].fillna(0)

        try:
            if hasattr(pitch_length_model, 'feature_names_in_'):
                X_len = pd.DataFrame([{k: row.get(k, 0.0) for k in pitch_length_model.feature_names_in_}])
            else:
                X_len = pd.DataFrame([row])[required].fillna(0)
        except Exception:
            X_len = pd.DataFrame([row])[required].fillna(0)

        # Predict safely
        try:
            pred_line = pitch_line_model.predict(X_line)[0] if pitch_line_model is not None else None
        except Exception:
            pred_line = None
        try:
            pred_len = pitch_length_model.predict(X_len)[0] if pitch_length_model is not None else None
        except Exception:
            pred_len = None

        try:
            line_label = pitchLine_mapping.get(int(pred_line), str(pred_line)) if pred_line is not None else 'Unknown'
        except Exception:
            line_label = str(pred_line)
        try:
            len_label = pitchLength_mapping.get(int(pred_len), str(pred_len)) if pred_len is not None else 'Unknown'
        except Exception:
            len_label = str(pred_len)

        # compute per-style approximate acc if df_line present
        def compute_acc(m, target_col):
            try:
                if m is None or df_line is None or df_line.empty or target_col not in df_line.columns:
                    return None
                sub = df_line[df_line.get('Bowler_Bowling_Style', '').astype(str).str.lower() == style.lower()]
                if sub.empty or len(sub) < 10:
                    return None
                feats = list(m.feature_names_in_) if hasattr(m, 'feature_names_in_') else [c for c in sub.select_dtypes(include='number').columns if c != target_col]
                X = sub[feats].fillna(0)
                y = sub[target_col]
                from sklearn.metrics import accuracy_score
                return float(accuracy_score(y, m.predict(X)))
            except Exception:
                return None

        acc_line = compute_acc(pitch_line_model, 'pitchLine_code')
        acc_len = compute_acc(pitch_length_model, 'pitchLength_code')

        results.append({
            'Bowling Type': style,
            'Predicted Line': line_label,
            'Predicted Length': len_label,
            'Line Model Accuracy': round(acc_line, 3) if acc_line is not None else 'NA',
            'Length Model Accuracy': round(acc_len, 3) if acc_len is not None else 'NA'
        })

    return {'Batsman': batsman_name, 'Phase': phase, 'Recommendations': results}


# ---------- Load batsman_stats ----------
batsman_group = safe_csv_load(FILES['batsman_stats'])
if batsman_group.empty:
    alt = DATA_DIR / 'batsman_stats1.csv'
    if alt.exists():
        batsman_group = safe_csv_load(alt)
if batsman_group.empty:
    print('[warn] batsman_stats not found or empty. Tactical recommendations that rely on this will be disabled.')
else:
    # normalize name column for matching
    batsman_group, _ = normalize_df_name_column(batsman_group)

# ---------- small helpers used by callbacks ----------

def _find_column(df, possible_names):
    if df is None or df.empty:
        return None
    lower = {c.lower(): c for c in df.columns}
    for p in possible_names:
        if p.lower() in lower:
            return lower[p.lower()]
    return None


def match_batsman(batsman_name, df):
    if df is None or df.empty or batsman_name is None:
        return None
    name_col = next((c for c in df.columns if 'name' in c.lower()), None)
    if name_col is None:
        return None
    df_loc = df.copy()
    df_loc['_clean'] = df_loc[name_col].astype(str).apply(lambda x: normalize_name_simple(x)).str.title()
    target = normalize_name_simple(batsman_name).title()
    exact = df_loc[df_loc['_clean'] == target]
    if not exact.empty:
        return exact.iloc[0]
    surname = target.split()[-1] if target else ''
    if surname:
        ends = df_loc[df_loc['_clean'].str.endswith(surname)]
        if not ends.empty:
            return ends.iloc[0]
    import difflib
    close = difflib.get_close_matches(target, df_loc['_clean'].tolist(), n=1, cutoff=0.6)
    if close:
        return df_loc[df_loc['_clean'] == close[0]].iloc[0]
    return None

# ---------- Plot helpers (unchanged) ----------

def make_overview_cards(df):
    if df is None or df.empty:
        return dbc.Row([dbc.Col(html.Div("No data loaded"))])
    matches = int(df['match_obj_id'].nunique()) if 'match_obj_id' in df.columns else (int(df['match_id'].nunique()) if 'match_id' in df.columns else "—")
    players = int(pd.concat([df.get('Batsman_Name', pd.Series(dtype=str)), df.get('Bowler', pd.Series(dtype=str))]).nunique())
    seasons = df['season'].nunique() if 'season' in df.columns else "—"
    total_runs = int(df['Runs'].sum()) if 'Runs' in df.columns else "—"
    cards = dbc.Row([
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Matches"), html.H3(matches)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Unique players"), html.H3(players)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Seasons"), html.H3(seasons)])), md=3),
        dbc.Col(dbc.Card(dbc.CardBody([html.H6("Total runs"), html.H3(total_runs)])), md=3),
    ], class_name='mb-3')
    return cards


def batsman_heatmap(df, batsman, bins_line=12, bins_length=12):
    if df is None or df.empty:
        return go.Figure(layout=go.Layout(title="No data"))
    if 'pitchLine' not in df.columns or 'pitchLength' not in df.columns:
        if 'x' in df.columns and 'y' in df.columns:
            sub = df[df.get('Batsman_Name') == batsman] if batsman else df.sample(min(2000, len(df)))
            return px.density_heatmap(sub, x='x', y='y', nbinsx=bins_line, nbinsy=bins_length, title=f"Shot density — {batsman}")
        return go.Figure(layout=go.Layout(title="Pitch data not present"))
    sub = df[df.get('Batsman_Name') == batsman] if batsman else df.sample(min(2000, len(df)))
    if sub.empty:
        sub = df.sample(min(2000, len(df)))
    fig = px.density_heatmap(sub, x='pitchLine', y='pitchLength', nbinsx=bins_line, nbinsy=bins_length, labels={'pitchLine': 'Pitch line', 'pitchLength': 'Pitch length'}, title=f"Pitch Map — {batsman or 'Overall'}")
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    return fig


def phase_strike_rate(df, batsman=None):
    if df is None or df.empty:
        return go.Figure(layout=go.Layout(title="No phase SR data"))
    cols = {c.lower(): c for c in df.columns}
    phase_col = cols.get('match_phase') or cols.get('phase')
    sr_col = cols.get('strike_rate') or cols.get('sr')
    if not phase_col or not sr_col:
        return go.Figure(layout=go.Layout(title="Phase or Strike Rate column missing"))
    # ensure batsman matching uses normalized values
    name_col = next((c for c in df.columns if 'name' in c.lower()), None)
    if batsman and name_col:
        lookup = normalize_name_simple(batsman).title()
        d = df[df[name_col].astype(str).str.contains(lookup, case=False, na=False)]
    else:
        d = df
    if d.empty:
        return go.Figure(layout=go.Layout(title=f"No SR data for {batsman}"))
    fig = px.bar(d.groupby(phase_col, as_index=False)[sr_col].mean(), x=phase_col, y=sr_col, color=phase_col, title=f"Phase-wise Strike Rate — {batsman or 'Overall'}")
    fig.update_layout(template="plotly_white", font=dict(family="Arial", size=13), margin=dict(l=20, r=20, t=40, b=20), showlegend=False)
    return fig


def top_similar_batsmen(sim_df, batsman, top_n=5):
    if sim_df is None or sim_df.empty:
        return go.Figure(layout=go.Layout(title="No similarity data"))
    sim_df.index = sim_df.index.astype(str).str.strip().str.title()
    sim_df.columns = sim_df.columns.astype(str).str.strip().str.title()
    lookup_name = batsman
    if batsman not in sim_df.index:
        surname = batsman.split()[-1].lower() if isinstance(batsman, str) and batsman.strip() else ""
        possible = [name for name in sim_df.index if surname and name.lower().endswith(surname)]
        if possible:
            lookup_name = possible[0]
    if lookup_name not in sim_df.index:
        return go.Figure(layout=go.Layout(title=f"{batsman} not found in similarity data"))
    s = sim_df.loc[lookup_name].dropna()
    s = s[s.index != lookup_name]
    s = s.sort_values(ascending=False).head(top_n)
    if s.empty:
        return go.Figure(layout=go.Layout(title=f"No similar batsmen found for {batsman}"))
    df = pd.DataFrame({"similar_batsman": s.index, "similarity_score": s.values})
    df["similarity_score"] = df["similarity_score"].round(3)
    fig = px.bar(df, x="similar_batsman", y="similarity_score", title=f"Top {top_n} similar batsmen to {lookup_name}", color="similarity_score")
    fig.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20), font=dict(family="Arial", size=13), showlegend=False)
    return fig


def top_bowlers_by_success(df_bowler, top_n=10):
    if df_bowler is None or df_bowler.empty:
        return go.Figure(layout=go.Layout(title="No bowler data"))
    si_col = next((c for c in df_bowler.columns if 'success' in c.lower()), None)
    if si_col is None:
        if 'totalWickets' in df_bowler.columns:
            si_col = 'totalWickets'
        else:
            numeric = df_bowler.select_dtypes(include='number').columns.tolist()
            si_col = numeric[0] if numeric else None
    if si_col is None:
        return go.Figure(layout=go.Layout(title="No metric columns in bowler stats"))
    d = df_bowler.sort_values(si_col, ascending=False).head(top_n)
    xcol = 'Bowler_Name' if 'Bowler_Name' in d.columns else (d.columns[0] if len(d.columns) > 0 else None)
    if xcol is None:
        return go.Figure(layout=go.Layout(title="No bowler name column"))
    fig = px.bar(d, x=xcol, y=si_col, title='Top bowlers by ' + si_col)
    fig.update_layout(margin=dict(l=20, r=20, t=40, b=20))
    return fig

# ---------- Layout ----------

batsmen = sorted(df_final['Batsman_Name'].dropna().unique()) if isinstance(df_final, pd.DataFrame) and 'Batsman_Name' in df_final.columns else []
batsman_options = [{'label': b, 'value': b} for b in batsmen]
if {'label':'Overall','value':'Overall'} not in batsman_options:
    batsman_options.insert(0, {'label': 'Overall', 'value': 'Overall'})

app = Dash(__name__, external_stylesheets=[dbc.themes.LUMEN], suppress_callback_exceptions=True)
app._callbacks = {}
app.title = "Cricket Analytics — Strategy Dashboard"
server = app.server

app.layout = dbc.Container([
    html.H2("Cricket Analytics — Strategy Dashboard", style={'marginTop': 10}),
    html.Hr(),
    make_overview_cards(df_final),
    dbc.Card([dbc.CardBody([
        dbc.Row([
            dbc.Col([html.Label('Batsman'), dcc.Dropdown(id='batsman-filter', options=batsman_options, value=batsmen[0] if batsmen else None)], md=4),
            dbc.Col([html.Label('Season'), dcc.Dropdown(id='season-filter', options=[{'label': s, 'value': s} for s in sorted(df_final['season'].dropna().unique())] if 'season' in df_final.columns else [], multi=True)], md=4),
            dbc.Col([html.Label('Team'), dcc.Dropdown(id='team-filter', options=[{'label': t, 'value': t} for t in sorted(df_final['Batting_Team'].dropna().unique())] if 'Batting_Team' in df_final.columns else [])], md=4),
        ])
    ])], class_name='mb-3'),

    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Batsman: Heatmap'), dbc.CardBody(dcc.Graph(id='heatmap-graph'))]), md=6),
        dbc.Col(dbc.Card([dbc.CardHeader('Phase-wise Strike Rate'), dbc.CardBody(dcc.Graph(id='phase-sr-graph'))]), md=6),
    ], class_name='mb-3'),

    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Top similar batsmen'), dbc.CardBody(dcc.Graph(id='similar-batsmen-graph'))]), md=6),
        dbc.Col(dbc.Card([dbc.CardHeader('Top bowlers by success'), dbc.CardBody(dcc.Graph(id='top-bowlers-graph'))]), md=6),
    ], class_name='mb-3'),

    dbc.Row([
        dbc.Col(dbc.Card([dbc.CardHeader('Tactical Recommendation'), dbc.CardBody(html.Div(id='table-preview'))]), md=8),
        dbc.Col(dbc.Card([dbc.CardHeader('Model predictions'), dbc.CardBody(html.Div(id='model-predictions'))]), md=4),
    ], class_name='mb-3')
], fluid=True)

# ---------- get_recommendation ----------

def get_recommendation(batsman_name, phase, bowler_type="right-arm fast"):
    try:
        if batsman_name is None or str(batsman_name).strip() == "":
            return None
        if 'batsman_group' not in globals() or batsman_group is None or batsman_group.empty:
            return None
        name_col = next((c for c in batsman_group.columns if 'name' in c.lower()), None)
        if name_col is None:
            return None
        candidates = batsman_group[batsman_group[name_col].astype(str).str.contains(str(batsman_name), case=False, na=False)]
        if candidates.empty:
            surname = str(batsman_name).split()[-1]
            candidates = batsman_group[batsman_group[name_col].astype(str).str.lower().str.endswith(surname.lower())]
        if candidates.empty:
            import difflib
            cleaned = batsman_group[name_col].astype(str).str.lower().tolist()
            matches = difflib.get_close_matches(str(batsman_name).lower(), cleaned, n=1, cutoff=0.6)
            if matches:
                idx = cleaned.index(matches[0])
                row = batsman_group.iloc[idx]
            else:
                row = None
        else:
            row = candidates.iloc[0]
        if row is None:
            return None
        adapt_col = next((col for col in ["consistency_index", "adaptability_index", "adaptability", "adaptability_score"] if col in batsman_group.columns), None)
        try:
            adaptability = float(row[adapt_col]) if adapt_col and pd.notna(row.get(adapt_col, None)) else 0.5
        except Exception:
            adaptability = 0.5

        strike_rate = 90.0
        try:
            if isinstance(phase_sr, pd.DataFrame) and not phase_sr.empty:
                pname_col = next((c for c in phase_sr.columns if 'name' in c.lower()), None)
                pphase_col = next((c for c in phase_sr.columns if 'phase' in c.lower()), None)
                psr_col = next((c for c in phase_sr.columns if 'strike' in c.lower() or 'sr' in c.lower()), None)
                if pname_col and psr_col:
                    sub = phase_sr[phase_sr[pname_col].astype(str).str.contains(str(batsman_name), case=False, na=False)]
                    if not sub.empty:
                        if pphase_col:
                            match_row = sub[sub[pphase_col].astype(str).str.strip().str.title() == str(phase).strip().title()]
                            if match_row.empty:
                                match_row = sub.iloc[[0]]
                        else:
                            match_row = sub.iloc[[0]]
                        strike_rate = float(match_row[psr_col].iloc[0]) if pd.notna(match_row[psr_col].iloc[0]) else strike_rate
        except Exception:
            pass

        try:
            aggressiveness = (strike_rate / 150.0) * (1 + (1 - float(adaptability)))
        except Exception:
            aggressiveness = (strike_rate / 150.0) * 1.0
        difficulty = "High Threat" if aggressiveness > 1.2 else ("Medium Threat" if aggressiveness > 0.9 else "Low Threat")

        if difficulty == "High Threat":
            if "fast" in bowler_type:
                line = "OUTSIDE_OFF"
                length = "SHORT_OF_A_GOOD_LENGTH"
                plan = "Keep it back of a length; attack with short balls and tight off-stump line."
            else:
                line = "ON_THE_STUMPS"
                length = "FULL"
                plan = "Use flight and drift; tempt drive — keep cover and long-off deep."
        elif difficulty == "Medium Threat":
            if adaptability > 0.6:
                line = "ON_THE_STUMPS"
                length = "GOOD_LENGTH"
                plan = "Stick to classic good-length channels; mix in slower balls."
            else:
                line = "OUTSIDE_OFF"
                length = "FULL"
                plan = "Pitch it fuller, outside off; use swing and variation."
        else:
            line = "ON_THE_STUMPS"
            length = "YORKER"
            plan = "Attack with full, straight deliveries; use yorkers and narrow lines."

        phase_title = str(phase).strip().title()
        if phase_title == "Powerplay":
            plan += " Restrict boundaries; field up close with attacking slips."
        elif phase_title == "Middle":
            plan += " Maintain pressure; rotate bowlers and change pace subtly."
        else:
            plan += " Focus on yorkers and wide lines; deep square leg and third man back."

        if "fast" in bowler_type:
            field_plan = "Deep square leg, third man, cover point, long-off, long-on."
        elif "offbreak" in bowler_type or "off" in bowler_type:
            field_plan = "Slip, short mid-wicket, long-on, long-off, deep mid-wicket."
        else:
            field_plan = "Balanced: cover, mid-off, deep mid-wicket, square leg."

        return {
            "Batsman": batsman_name,
            "Phase": phase,
            "Bowler Type": bowler_type,
            "Adaptability Index": round(float(adaptability), 3),
            "Strike Rate": round(float(strike_rate), 1),
            "Threat Level": difficulty,
            "Recommended Line": line,
            "Recommended Length": length,
            "Tactical Plan": plan,
            "Field Setup": field_plan
        }
    except Exception as _e:
        print(f"[warn] get_recommendation failed: {_e}")
        return None

# ---------- Callbacks ----------
@app.callback(
    Output("heatmap-graph", "figure"),
    Output("phase-sr-graph", "figure"),
    Output("similar-batsmen-graph", "figure"),
    Output("top-bowlers-graph", "figure"),
    Output("table-preview", "children"),
    Output("model-predictions", "children"),
    Input("batsman-filter", "value"),
    Input("season-filter", "value"),
    Input("team-filter", "value"),
)

def update_dashboard(batsman, seasons, team):
    try:
        if batsman in [None, '', 'Overall']:
            batsman_val = df_final['Batsman_Name'].dropna().iloc[0] if isinstance(df_final, pd.DataFrame) and 'Batsman_Name' in df_final.columns and not df_final.empty else None
        else:
            batsman_val = batsman

        d = df_final.copy() if isinstance(df_final, pd.DataFrame) and not df_final.empty else pd.DataFrame()
        if seasons and 'season' in d.columns:
            d = d[d['season'].isin(seasons)]
        if team and 'Batting_Team' in d.columns:
            d = d[d['Batting_Team'] == team]

        try:
            heat = batsman_heatmap(d if not d.empty else df_final, batsman_val)
        except Exception as e:
            print('Heatmap error:', e)
            heat = go.Figure()

        try:
            phase = phase_strike_rate(phase_sr, batsman_val)
        except Exception as e:
            print('Phase SR error:', e)
            phase = go.Figure()

        try:
            sim = top_similar_batsmen(batsman_similarity, batsman_val)
        except Exception as e:
            print('Similarity error:', e)
            sim = go.Figure()

        try:
            bowl = top_bowlers_by_success(bowling_success)
        except Exception as e:
            print('Bowlers error:', e)
            bowl = go.Figure()

        if d is None or d.empty or 'oversActual' not in d.columns:
            phase_guess = "Middle"
        else:
            try:
                ov = pd.to_numeric(d['oversActual'], errors='coerce').mean()
                if np.isnan(ov):
                    phase_guess = "Middle"
                else:
                    phase_guess = "Powerplay" if ov <= 6 else "Middle" if ov <= 15 else "Death"
            except Exception:
                phase_guess = "Middle"

        rec = get_recommendation(batsman_val, phase_guess, bowler_type="right-arm fast") if batsman_val else None

        if rec is None:
            table_html = html.Div("No tactical data available for this batsman.")
        else:
            tactical_children = [
                html.P(f"Threat Level: {rec['Threat Level']}"),
                html.P(f"Adaptability Index: {rec['Adaptability Index']}"),
                html.P(f"Strike Rate: {rec['Strike Rate']}"),
                html.H6("Recommended Line & Length:"),
                html.P(f"{rec['Recommended Line']} — {rec['Recommended Length']}"),
                html.H6("Tactical Plan:"),
                html.P(rec["Tactical Plan"]),
                html.H6("Field Setup:"),
                html.P(rec["Field Setup"]),
                html.Hr()
            ]

            try:
                opt = recommend_optimal_strategy(batsman_val, phase_guess)
                if isinstance(opt, dict) and opt.get("Recommendations"):
                    tactical_children.append(html.H5("AI Optimal Strategy (Model-Based)"))
                    for r in opt["Recommendations"]:
                        tactical_children.extend([
                            html.H6(f"Bowling Type: {r.get('Bowling Type', 'Unknown')}"),
                            html.P(f"Predicted Line: {r.get('Predicted Line', '')}"),
                            html.P(f"Predicted Length: {r.get('Predicted Length', '')}"),
                            html.P(f"Line Model Accuracy: {r.get('Line Model Accuracy', 'NA')}"),
                            html.P(f"Length Model Accuracy: {r.get('Length Model Accuracy', 'NA')}"),
                            html.Hr()
                        ])
                else:
                    tactical_children.append(html.Div("AI model recommendations not available (models or features missing)."))
            except Exception as e:
                print("Error generating AI optimal strategy:", e)
                tactical_children.append(html.Div("AI optimal strategy failed (see logs)."))

            table_html = dbc.Card([dbc.CardHeader(f"Tactical Bowling Plan Against {rec['Batsman']} — {rec['Phase']} Phase"), dbc.CardBody(tactical_children)])

        model_div = html.Div("No model available")
        if predict_good_ball_model is not None:
            try:
                df_engineered = MODEL_DATASETS.get("good_ball", pd.DataFrame()).copy()
                if batsman and batsman not in [None, '', 'Overall'] and 'Batsman_Name' in df_engineered.columns:
                    df_engineered = df_engineered[df_engineered['Batsman_Name'].str.contains(batsman, case=False, na=False)]
                if team and 'Batting_Team' in df_engineered.columns:
                    df_engineered = df_engineered[df_engineered['Batting_Team'] == team]
                if seasons and 'season' in df_engineered.columns:
                    df_engineered = df_engineered[df_engineered['season'].isin(seasons)]

                if df_engineered.empty:
                    model_div = html.Div("No data available for this selection.")
                else:
                    required_features = list(predict_good_ball_model.feature_names_in_) if hasattr(predict_good_ball_model, "feature_names_in_") else df_engineered.select_dtypes(include='number').columns.tolist()
                    for col in required_features:
                        if col not in df_engineered.columns:
                            df_engineered[col] = 0.0

                    df_encoded = df_engineered.copy()
                    for col in df_encoded.select_dtypes(include='object').columns:
                        df_encoded[col] = df_encoded[col].astype('category').cat.codes

                    X = df_encoded.reindex(columns=required_features, fill_value=0)
                    if hasattr(predict_good_ball_model, 'predict_proba'):
                        df_engineered['good_ball_prob'] = predict_good_ball_model.predict_proba(X)[:, 1]
                    else:
                        df_engineered['good_ball_prob'] = predict_good_ball_model.predict(X)

                    avg_prob = df_engineered['good_ball_prob'].mean()
                    top10 = df_engineered.nlargest(10, 'good_ball_prob')

                    prob_chart = px.histogram(df_engineered, x='good_ball_prob', nbins=20, title=f"Distribution of Good Ball Probabilities — {batsman or 'All Batsmen'}")
                    donut_chart = go.Figure(go.Indicator(mode="gauge+number", value=avg_prob * 100, title={'text': f"Avg Good Ball % — {batsman or 'Overall'}"}, gauge={'axis': {'range': [0, 100]}}))
                    top_chart = px.bar(top10, y='good_ball_prob', title=f"Top 10 Balls with Highest Good-Ball Probability — {batsman or 'All Batsmen'}", color='good_ball_prob')

                    model_div = html.Div([
                        html.H6(f"Model Prediction Summary — {batsman or 'Overall'}"),
                        html.P(f"Average Good Ball Probability: {avg_prob:.2f}"),
                        dbc.Row([
                            dbc.Col(dcc.Graph(figure=donut_chart), md=4),
                            dbc.Col(dcc.Graph(figure=prob_chart), md=8),
                        ]),
                        html.Hr(),
                        dcc.Graph(figure=top_chart)
                    ])
            except Exception as e:
                print("Model prediction error:", e)
                print(traceback.format_exc())
                model_div = html.Div("Model error: see server logs")

        return heat, phase, sim, bowl, table_html, model_div

    except Exception as e:
        print("\n\n===== DASH CALLBACK CRASHED =====")
        print(e)
        print(traceback.format_exc())
        print("=================================\n\n")
        return go.Figure(), go.Figure(), go.Figure(), go.Figure(), html.Div("Error generating tactical recommendation"), html.Div("Error")

# ---------- Run ----------
if __name__ == '__main__':
    print('Starting enhanced dashboard at http://127.0.0.1:8050')
    app.run(debug=True)


[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets\df_model_good_ball.csv -> (27273, 10)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets\df_model_ball_type.csv -> (33029, 12)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets\df_model_boundary.csv -> (27273, 10)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\model_feature_engineered_datasets\df_model_line.csv -> (74, 13)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\final_processed_data.csv -> (33029, 30)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_profiles.csv -> (462, 8)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\artificial intelligence project\data\batsman_similarity.csv -> (462, 462)
[info] Loaded C:\Users\yaswa\OneDrive\Desktop\arti

In [2]:
print("Similarity shape:", batsman_similarity.shape)
print("First 10 index names:", list(batsman_similarity.index[:10]))
print("First 10 column names:", list(batsman_similarity.columns[:10]))
print("Does 'A Balbirnie' exist as index?:", 'A Balbirnie' in batsman_similarity.index)
print("Does 'A Balbirnie' exist as column?:", 'A Balbirnie' in batsman_similarity.columns)


Similarity shape: (462, 462)
First 10 index names: ['Ab De Villiers', 'Aamir Kaleem', 'Aaron Finch', 'Aaron Johnson', 'Aaron Jones', 'Aaron Phangiso', 'Aasif Sheikh', 'Aayan Afzal Khan', 'Abinash Bohara', 'Adam Milne']
First 10 column names: ['Ab De Villiers', 'Aamir Kaleem', 'Aaron Finch', 'Aaron Johnson', 'Aaron Jones', 'Aaron Phangiso', 'Aasif Sheikh', 'Aayan Afzal Khan', 'Abinash Bohara', 'Adam Milne']
Does 'A Balbirnie' exist as index?: False
Does 'A Balbirnie' exist as column?: False


In [3]:
name = "A Vala"
matches = batsman_group[batsman_group["Batsman_Name"].str.contains(name, case=False, na=False)]
print("Matches found:", len(matches))
print(matches.head())


KeyError: 'Batsman_Name'